# Lección 4.5 — El compromiso sesgo-varianza: la analogía del tirador al blanco

> **Módulo 4 · Evaluación: ¿mi modelo es bueno?** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 4.1 (MSE/RMSE), Lección 4.4 (sobreajuste vs. subajuste)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con la analogía del tirador qué son el **sesgo** y la **varianza** de un modelo.
- Entender por qué el error de un modelo se reparte en **sesgo² + varianza + ruido**.
- Predecir qué perillas bajan el sesgo y cuáles bajan la varianza (y por qué no puedes bajar las dos gratis).
- Diagnosticar, con datos reales, si tu modelo sufre más de sesgo o de varianza.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.model_selection import train_test_split

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el tirador al blanco 🎯

Imagina a cuatro tiradores. Cada uno dispara **20 veces** a una diana. Al final miramos dos cosas:

- **¿Dónde cae el centro de sus tiros?** Si el promedio de sus tiros está lejos del centro de la diana, el tirador tiene un **error sistemático**: quizá la mira del rifle está chueca. Eso es el **sesgo**.
- **¿Qué tan dispersos están sus tiros?** Si cada tiro cae en un lugar muy distinto, el tirador es **inconsistente**: le tiembla el pulso. Eso es la **varianza**.

Son dos defectos **distintos**: un tirador puede tener la mira chueca pero un pulso perfecto (todos los tiros juntitos… en el lugar equivocado), o la mira perfecta y un pulso tembloroso (tiros alrededor del centro, pero desparramados).

▶️ Ejecuta para ver a los cuatro tiradores.

In [ ]:
rng = np.random.default_rng(3)
tiradores = {   # nombre: (centro de sus tiros, qué tanto le tiembla el pulso)
    "A": ((0.9, 0.7), 0.15),
    "B": ((0.0, 0.0), 0.15),
    "C": ((0.9, 0.7), 0.55),
    "D": ((0.0, 0.0), 0.55),
}
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (nombre, (centro, pulso)) in zip(axes, tiradores.items()):
    for radio in [2, 1.5, 1, 0.5]:
        ax.add_patch(plt.Circle((0, 0), radio, fill=False, color="gray"))
    ax.plot(0, 0, "r+", ms=15, mew=2)
    tiros = rng.normal(centro, pulso, size=(20, 2))
    ax.scatter(*tiros.T, s=25, c="k")
    ax.set_xlim(-2.1, 2.1); ax.set_ylim(-2.1, 2.1); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"Tirador {nombre}")
plt.show()

### ✋ Ejercicio 1 — Diagnostica a los tiradores
Para cada tirador, di si tiene **sesgo alto o bajo** y **varianza alta o baja**:

| Tirador | Sesgo (alto/bajo) | Varianza (alta/baja) |
|---|---|---|
| A | | |
| B | | |
| C | | |
| D | | |

¿Cuál contratarías? ¿Y a cuál le arreglarías la mira y a cuál le darías clases para calmar el pulso?

<details><summary><b>👀 Ver respuesta</b></summary>

| Tirador | Sesgo | Varianza |
|---|---|---|
| A | **alto** (todos juntos, pero arriba a la derecha) | **baja** |
| B | **bajo** | **baja** → el ideal |
| C | **alto** | **alta** → el peor |
| D | **bajo** (en promedio apunta al centro) | **alta** |

Contratarías a **B**. A **A** le arreglarías la mira (corregir el sesgo); a **D** le calmarías el pulso (reducir la varianza). Fíjate en que **D** acierta "en promedio", pero cualquier tiro individual puede caer lejísimos: eso también es un problema.
</details>

---
## 2. 📖 El concepto: ¿quién es el tirador y qué son los tiros?

En ML, el **tirador** es tu tipo de modelo (por ejemplo, "una recta" o "un árbol de profundidad 10") y cada **tiro** es el modelo que obtienes al entrenarlo con **un conjunto de datos concreto**.

Si hubieras recolectado **otra muestra** de datos del mismo fenómeno (otros 30 clientes, otros 30 días), habrías entrenado un modelo un poco distinto: otro tiro.

| Concepto | En la diana | En ML |
|---|---|---|
| **Sesgo** (*bias*) | El promedio de los tiros está lejos del centro | Incluso promediando muchos modelos entrenados con muestras distintas, el modelo se equivoca **siempre en la misma dirección**. El modelo es demasiado **rígido** para la forma real de los datos. |
| **Varianza** (*variance*) | Los tiros están desparramados | El modelo **cambia mucho** según la muestra con la que lo entrenaste. Es demasiado **sensible** a los detalles (y al ruido) de cada muestra. |
| **Ruido irreducible** | El viento: mueve cualquier tiro, por bueno que sea el tirador | La parte de la etiqueta que **ninguna** característica explica (azar, errores de medición). |

> 🔗 Conexión con la Lección 4.4: **subajuste ≈ sesgo alto** (el modelo no captura el patrón) y **sobreajuste ≈ varianza alta** (el modelo memoriza el ruido de *su* muestra).

### ✋ Ejercicio 2 — ¿Dónde está la varianza?
Un compañero dice: *"Mi modelo no puede tener varianza: yo solo tengo **un** conjunto de datos y entrené **un** modelo."*

¿Tiene razón? ✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** La varianza no es algo que veas en tu único modelo: es una propiedad de **cómo habría salido** si tus datos hubieran sido otros. Tu conjunto de datos es una de muchas muestras posibles; tu modelo es **un tiro**. Si el modelo es muy sensible, ese único tiro pudo caer lejos del centro sin que te enteres… hasta que llegan datos nuevos.

En la vida real no tenemos 200 conjuntos de datos, pero en esta lección **sí podemos simularlos**, porque vamos a inventar el fenómeno nosotros.
</details>

---
## 3. 📖 Cómo se ve: muchos tiros de un mismo modelo

Inventamos un fenómeno con forma de **ola** (la curva verdadera, que en la vida real nunca conocerías) y le sumamos ruido. Luego sacamos **varias muestras de 30 puntos** del mismo fenómeno. Cada muestra es "los datos que te tocaron".

In [ ]:
def verdad(x):
    return np.sin(2 * np.pi * x)          # la curva real (el "centro de la diana")

RUIDO = 0.3                                # ✏️ qué tanto "viento" hay
x_malla = np.linspace(0.05, 0.95, 50)     # puntos donde evaluaremos los modelos

def nueva_muestra(rng, n=30, ruido=None):
    ruido = RUIDO if ruido is None else ruido    # si no se indica, usa el RUIDO actual
    x = rng.uniform(0, 1, n)
    return x, verdad(x) + rng.normal(0, ruido, n)

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for i, ax in enumerate(axes):
    x, y = nueva_muestra(rng)
    ax.plot(x_malla, verdad(x_malla), "g--", label="verdad")
    ax.scatter(x, y, s=20, c="k")
    ax.set_title(f"Muestra {i + 1}: los datos que te tocaron")
axes[0].legend(); plt.show()

Las tres muestras salen del **mismo** fenómeno, pero cada una tiene puntos en sitios distintos y ruido distinto.

Ahora el experimento clave: entrenamos el **mismo tipo de modelo** en **50 muestras distintas** y dibujamos los 50 modelos encima. Usaremos regresiones polinómicas como en la Lección 4.4: el **grado** es la flexibilidad del modelo (grado 1 = recta).

In [ ]:
def polinomio(grado):
    return make_pipeline(PolynomialFeatures(grado), LinearRegression())

def simular(crear_modelo, n=30, repeticiones=200, ruido=None, semilla=0):
    """Entrena el modelo en muchas muestras distintas.
    Devuelve una matriz: una fila de predicciones (sobre x_malla) por cada muestra."""
    rng = np.random.default_rng(semilla)
    predicciones = []
    for _ in range(repeticiones):
        x, y = nueva_muestra(rng, n, ruido)
        modelo = crear_modelo().fit(x.reshape(-1, 1), y)
        predicciones.append(modelo.predict(x_malla.reshape(-1, 1)))
    return np.array(predicciones)

def dibujar_tiros(grados):
    fig, axes = plt.subplots(1, len(grados), figsize=(4.7 * len(grados), 3.8), sharey=True, squeeze=False)
    axes = axes[0]
    for ax, g in zip(axes, grados):
        P = simular(lambda: polinomio(g), repeticiones=50)
        ax.plot(x_malla, P.T, color="steelblue", alpha=0.25)          # cada línea = un "tiro"
        ax.plot(x_malla, P.mean(axis=0), "orange", lw=3, label="promedio de los tiros")
        ax.plot(x_malla, verdad(x_malla), "g--", lw=2, label="verdad")
        ax.set_ylim(-2.5, 2.5); ax.set_title(f"Grado {g}: 50 modelos")
    axes[0].legend(loc="lower left"); plt.show()

dibujar_tiros([1])

La recta (grado 1) **no puede** doblarse como la ola: todos sus tiros se parecen entre sí (poca dispersión), pero su promedio (naranja) está lejos de la verdad (verde). **Sesgo alto, varianza baja**: el tirador A.

### 🔮 Ejercicio 3 — Predice la nube
Ahora dibujaremos los grados **3** y **12**. Antes de ejecutar:
1. ¿Cuál tendrá el promedio (naranja) más pegado a la verdad?
2. ¿Cuál tendrá la nube de líneas azules más **dispersa**?
3. ¿A qué tirador (A, B, C o D) se parece cada uno?

✍️ *Mis predicciones:*

In [ ]:
dibujar_tiros([1, 3, 12])

<details><summary><b>👀 Ver respuesta</b></summary>

1. Los grados **3 y 12** tienen el promedio casi encima de la verdad: los dos son lo bastante flexibles para la ola → **sesgo bajo**.
2. El grado **12** es un desastre de dispersión: cada muestra produce una curva salvaje, sobre todo en los bordes, donde hay pocos puntos → **varianza alta**.
3. Grado 1 ≈ **tirador A** (sesgo alto, varianza baja). Grado 3 ≈ **tirador B** (el ideal). Grado 12 ≈ **tirador D** (acierta en promedio, pero cada tiro individual es poco fiable).

Lo inquietante del grado 12: **tú solo tienes una de esas líneas azules** y no sabes cuál.
</details>

---
## 4. 📖 La descomposición: el error se reparte en tres

Con las 200 simulaciones podemos **medir** cada pieza (con el error cuadrático, MSE, de la Lección 4.1):

- **Sesgo²** = qué tan lejos está el **promedio** de los modelos de la verdad (elevado al cuadrado).
- **Varianza** = qué tanto se **dispersan** los modelos alrededor de su propio promedio.
- **Ruido²** = la varianza del ruido (aquí, 0.3² = 0.09). Ningún modelo lo puede predecir.

Y se cumple, en promedio:

$$\text{error esperado con datos nuevos} \approx \text{sesgo}^2 + \text{varianza} + \text{ruido}^2$$

La celda siguiente calcula las tres piezas y, por separado, el **error real** que cometen los modelos contra datos nuevos con ruido, para comprobar que la suma cuadra.

In [ ]:
def descomponer(crear_modelo, n=30, ruido=None, semilla=0):
    ruido = RUIDO if ruido is None else ruido
    P = simular(crear_modelo, n=n, ruido=ruido, semilla=semilla)
    promedio = P.mean(axis=0)
    sesgo2 = np.mean((promedio - verdad(x_malla)) ** 2)
    varianza = np.mean(P.var(axis=0))
    # error real: comparamos cada modelo con datos NUEVOS (verdad + ruido fresco)
    rng = np.random.default_rng(99)
    y_nuevos = verdad(x_malla) + rng.normal(0, ruido, P.shape)
    error_real = np.mean((P - y_nuevos) ** 2)
    return sesgo2, varianza, error_real

filas = []
for g in [1, 2, 3, 4, 5, 7, 9, 12]:
    s2, v, err = descomponer(lambda: polinomio(g))
    filas.append({"grado": g, "sesgo²": s2, "varianza": v, "ruido²": RUIDO**2,
                  "suma": s2 + v + RUIDO**2, "error real": err})
tabla = pd.DataFrame(filas).set_index("grado").round(3)
tabla

### 🔮 Ejercicio 4 — Lee la tabla
1. ¿En qué grado baja el sesgo de golpe? ¿Por qué justo ahí? (Pista: mira la forma de la ola.)
2. ¿Qué le pasa a la varianza a partir del grado 7?
3. ¿Qué grado tiene el menor **error real**? ¿Coincide con el que tiene menor sesgo?
4. ¿Se parecen la columna "suma" y la columna "error real"?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Entre el grado **2 y 3** el sesgo² cae de ≈0.15 a ≈0.003. La ola sube, baja y vuelve a subir: necesita al menos una curva "en S", que una parábola (grado 2) no puede hacer pero un polinomio de grado 3 sí.
2. Se dispara: ≈0.02 en grado 5, ≈0.20 en grado 7 y **≈1.2** en grado 12.
3. El mínimo de error real está en los grados **3–5** (≈0.11). El grado 12 tiene el sesgo igual de bajo, pero el **peor** error de todos (≈1.3) por culpa de la varianza. **Bajo sesgo no basta.**
4. Sí, casi idénticas: la descomposición funciona. Las pequeñas diferencias son azar de la simulación.

Fíjate en que **ningún** grado baja de 0.09: ese es el piso del ruido.
</details>

In [ ]:
ax = tabla[["sesgo²", "varianza", "error real"]].plot(marker="o", logy=True)
ax.axhline(RUIDO**2, ls=":", c="gray", label="ruido² (piso imposible de bajar)")
ax.set_xlabel("grado del polinomio (complejidad →)"); ax.set_ylabel("error (escala log)")
ax.legend(); ax.set_title("El compromiso sesgo-varianza"); plt.show()

Esta es **la gráfica más famosa del ML**: al subir la complejidad, el sesgo baja y la varianza sube. El error total tiene forma de **U** y el mejor modelo está en el fondo de la U, donde **ninguno** de los dos es cero. Por eso se llama **compromiso** (*trade-off*): casi nunca puedes bajar uno sin subir el otro.

### ✏️ Ejercicio 5 — Más viento
Vuelve a la celda de la Sección 3 y cambia `RUIDO = 0.3` por `RUIDO = 0.6`. Luego vuelve a ejecutar la celda de la tabla y la gráfica.
- ¿Qué pasa con el piso del ruido?
- ¿Sigue ganando el mismo grado?

✍️ *Mis observaciones:*

> ⚠️ Cuando termines, **devuelve `RUIDO = 0.3`** y vuelve a ejecutar esa celda: el resto de la lección asume ese valor.

<details><summary><b>👀 Ver respuesta</b></summary>

- El piso sube de 0.09 a **0.36** (0.6²): ningún modelo puede bajar de ahí, por bueno que sea.
- El sesgo² casi no cambia, pero la varianza de **todos** los modelos crece, y mucho más la de los flexibles (grado 12: ≈1.2 → ≈4.8): hay más ruido que memorizar. Antes los grados 3, 4 y 5 empataban (≈0.11); ahora el más simple de ellos, el **grado 3** (≈0.41), gana con claridad. Regla práctica: **cuanto más ruidosos tus datos, más sencillo debe ser tu modelo.**
</details>

---
## 5. 📖 ¿Qué pasa si…? Dos armas contra la varianza

### 5.1 …consigo más datos?
Con más puntos por muestra, cada modelo tiene menos espacio para "inventar" curvas raras: los tiros se juntan.

### 🔮 Ejercicio 6
Tomamos el polinomio de **grado 9** (varianza alta con 30 puntos) y lo entrenamos con muestras de **30, 100 y 300** puntos. ¿Qué crees que pasa con el sesgo²? ¿Y con la varianza?

✍️ *Mi predicción:*

In [ ]:
for n in [30, 100, 300]:      # ✏️ prueba también n = 1000
    s2, v, err = descomponer(lambda: polinomio(9), n=n)
    print(f"n = {n:>4}  →  sesgo² = {s2:.3f}   varianza = {v:.3f}   error real = {err:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El **sesgo² casi no cambia** (ya era ≈0: el grado 9 es flexible de sobra). La **varianza se desploma**: ≈0.25 con 30 puntos, ≈0.007 con 100 y ≈0.002 con 300. El error real se acerca al piso de 0.09.

Conclusión: **más datos curan la varianza, no el sesgo.** Si tu modelo es una recta y el fenómeno es una ola, un millón de datos te dará… una recta muy segura de sí misma y igual de equivocada.
</details>

### 5.2 …promedio muchos modelos?
Si cada tiro tiembla en una dirección distinta, **promediar varios tiros** cancela parte del temblor. Esa es la idea del **bagging** (Módulo 6): entrenar muchos árboles, cada uno con un remuestreo distinto de los datos, y promediarlos.

### 🔮 Ejercicio 7
Comparamos un árbol de decisión **sin límite de profundidad** (Lección 5.4: sobreajusta, varianza alta) con el **promedio de 50 árboles iguales** entrenados con remuestreos de los mismos datos (`BaggingRegressor`). ¿Qué pieza bajará más: el sesgo² o la varianza?

✍️ *Mi predicción:*

In [ ]:
arbol = lambda: DecisionTreeRegressor(random_state=0)
bolsa = lambda: BaggingRegressor(DecisionTreeRegressor(), n_estimators=50, random_state=0)

for nombre, crear in [("1 árbol sin límite", arbol), ("promedio de 50 árboles", bolsa)]:
    s2, v, err = descomponer(crear)
    print(f"{nombre:<24} sesgo² = {s2:.3f}   varianza = {v:.3f}   error real = {err:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

La **varianza** baja a la mitad (≈0.10 → ≈0.05) y el sesgo² no cambia (≈0.001). El error real mejora de ≈0.19 a ≈0.14 **sin tocar la profundidad del árbol**.

Promediar es la gran trampa legal contra la varianza: por eso los ensambles (Random Forest, Módulo 6) dominan los datos tabulares.
</details>

---
## 6. 📖 Perillas: qué mueve el sesgo y qué mueve la varianza

Casi todos los hiperparámetros del curso son, en el fondo, una perilla de **complejidad**: la giras hacia un lado y bajas el sesgo; hacia el otro y bajas la varianza.

| Perilla | Para **bajar el sesgo** (modelo más flexible) | Para **bajar la varianza** (modelo más estable) |
|---|---|---|
| Grado del polinomio | Subirlo | Bajarlo |
| `max_depth` del árbol (5.4) | Subirlo | Bajarlo |
| `min_samples_leaf` del árbol (5.4) | Bajarlo | Subirlo |
| `k` de KNN (5.3) | **Bajarlo** | **Subirlo** |
| Regularización (`alpha`, `C`…, M7) | Menos regularización | Más regularización |
| Características | Añadir características útiles | Quitar características ruidosas |
| Cantidad de datos | — (no ayuda al sesgo) | **Más datos** |
| Promediar modelos (bagging, M6) | — | Promediar muchos |
| Boosting (M6) | Encadenar modelos que corrigen errores | — |

> 🧠 **Regla mnemotécnica:** *"Rígido → sesgo. Nervioso → varianza."*

### 🔮 Ejercicio 8 — KNN al revés
En KNN (Lección 5.3) el modelo predice promediando a los **k vecinos** más cercanos. ¿Con `k = 1` esperas sesgo o varianza alta? ¿Y con `k = 20`?

✍️ *Mi predicción:*

In [ ]:
for k in [1, 3, 5, 10, 20]:     # ✏️
    s2, v, err = descomponer(lambda: KNeighborsRegressor(n_neighbors=k))
    print(f"k = {k:>2}  →  sesgo² = {s2:.3f}   varianza = {v:.3f}   error real = {err:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `k = 1`: copia al vecino más cercano, ruido incluido → **varianza alta** (≈0.10), sesgo ≈0.
- `k = 20`: promedia a 20 de 30 puntos, casi el promedio de todos → la ola queda aplanada: **sesgo alto** (≈0.21).
- El fondo de la U está hacia **k = 3–5** (error ≈0.12–0.13), donde las dos piezas están equilibradas.

Ojo: en KNN la perilla funciona **al revés** que el grado del polinomio: **subir k simplifica**.
</details>

---
## 7. 📖 En la vida real no tienes 200 muestras: ¿cómo diagnosticas?

Hasta aquí hicimos trampa: inventamos el fenómeno y generamos todas las muestras que quisimos. Con datos reales tienes **una** muestra. La forma práctica de diagnosticar es comparar el error de **entrenamiento** con el de **validación** (Lección 4.4):

| Lo que ves | Diagnóstico | Qué probar |
|---|---|---|
| Error de entrenamiento **alto** y validación parecida (también alta) | **Sesgo alto** (subajuste) | Modelo más flexible, mejores características, menos regularización |
| Error de entrenamiento **bajo** y validación **mucho** más alta | **Varianza alta** (sobreajuste) | Más datos, modelo más simple, más regularización, promediar modelos |
| Los dos bajos y parecidos, cerca de lo que cabe esperar del problema | 🎉 Buen equilibrio | Validar con cuidado (Lección 4.6) y comparar con una línea base (Lección 4.7) |
| Los dos altos **y** brecha grande | Ambos problemas a la vez | Primero ataca el sesgo (mejores características), luego la varianza |

### 🚨 Errores comunes y señales de alerta
| 🚨 Error | Por qué está mal |
|---|---|
| "Conseguir más datos siempre mejora el modelo" | Solo si el problema es la **varianza**. Con sesgo alto, más datos no cambian nada. |
| "Si el error de entrenamiento es 0, el modelo es perfecto" | Es justo la firma de **varianza alta**: memorizó su muestra. |
| Buscar error 0 en validación | El **ruido irreducible** pone un piso; perseguirlo lleva a sobreajustar. |
| Subir la complejidad cuando el problema son los datos ruidosos | Con más ruido, el punto óptimo se mueve hacia modelos **más simples**. |
| Confiar en un único resultado de validación | Ese número también es "un tiro": tiene varianza. La validación cruzada (Lección 4.6) lo estabiliza. |

### 🔮 Ejercicio 9 — Médico de modelos
Diagnostica cada caso (sesgo alto, varianza alta o equilibrio) y propón **una** acción:

| Caso | RMSE entrenamiento | RMSE validación |
|---|---|---|
| 1 | 2.1 | 9.8 |
| 2 | 8.7 | 9.0 |
| 3 | 3.0 | 3.4 |

✍️ *Mis diagnósticos:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Varianza alta**: entrena muy bien y valida muy mal. Acciones: más datos, modelo más simple (bajar `max_depth`, subir `k`…), regularizar o promediar modelos.
2. **Sesgo alto**: ni siquiera aprende bien los datos que ya vio. Acciones: un modelo más flexible o mejores características. Más datos **no** ayudarían.
3. **Equilibrio razonable**: los dos bajos y parecidos. Siguiente paso: validación cruzada y comparar con una línea base.

Ojo: "alto" o "bajo" siempre es **relativo al problema**. Un RMSE de 3 es excelente prediciendo precios en millones y terrible prediciendo una nota de 0 a 10. Compara siempre contra una línea base (Lección 4.7).
</details>

▶️ Veámoslo en la práctica con **una sola** muestra (más grande) de nuestra ola: entrenamiento vs. validación para varios grados.

In [ ]:
rng = np.random.default_rng(44)
x, y = nueva_muestra(rng, n=60)
x_tr, x_va, y_tr, y_va = train_test_split(x.reshape(-1, 1), y, test_size=0.5, random_state=0)

for g in [1, 3, 12]:
    m = polinomio(g).fit(x_tr, y_tr)
    mse_tr = np.mean((m.predict(x_tr) - y_tr) ** 2)
    mse_va = np.mean((m.predict(x_va) - y_va) ** 2)
    print(f"grado {g:>2}  →  MSE entrenamiento = {mse_tr:7.3f}   MSE validación = {mse_va:9.3f}")

### ✋ Ejercicio 10
Sin mirar las simulaciones de antes, solo con estos tres renglones: ¿qué grado tiene sesgo alto, cuál varianza alta y cuál está equilibrado?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Grado 1**: entrenamiento y validación **los dos altos** (≈0.37 y ≈0.28) → **sesgo alto**: ni siquiera aprende bien los datos que ya vio.
- **Grado 12**: el entrenamiento es el más bajo de todos (≈0.07), pero la validación se dispara (≈1.08, más de 15 veces más) → **varianza alta**. Con solo 30 puntos de entrenamiento, el polinomio hace curvas salvajes en las zonas donde no hay datos.
- **Grado 3**: los dos bajos y del orden del ruido (≈0.09 y ≈0.17) → **equilibrado**.

(Que la validación del grado 1 salga un poco *menor* que su entrenamiento es pura suerte de qué puntos cayeron en cada mitad: con 30 puntos, estos números también tienen varianza.)

Llegamos a la misma conclusión que con 200 simulaciones, pero usando solo **una** muestra. Esta es la herramienta que usarás en la vida real.
</details>

---
## 8. 🧠 Resumen en 5 puntos
1. **Sesgo** = error sistemático: el modelo es demasiado **rígido** y falla siempre hacia el mismo lado (subajuste).
2. **Varianza** = inestabilidad: el modelo es demasiado **nervioso** y cambia mucho según la muestra de entrenamiento (sobreajuste).
3. Error esperado ≈ **sesgo² + varianza + ruido²**. El ruido es un piso que ningún modelo puede bajar.
4. Al subir la complejidad, el sesgo baja y la varianza sube: el mejor modelo está en el **fondo de la U**, no en un extremo.
5. **Más datos y promediar modelos** bajan la varianza; **modelos más flexibles y mejores características** bajan el sesgo. En la práctica, diagnosticas comparando el error de **entrenamiento** con el de **validación**.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** Un modelo tiene error de entrenamiento 0.40 y de validación 0.42; una línea base simple tiene 0.45. ¿Qué es lo más probable?
- a) Varianza alta · b) Sesgo alto · c) Fuga de datos · d) Ruido igual a cero

**2.** Tu modelo sobreajusta. ¿Cuál de estas acciones **no** ayuda a bajar la varianza?
- a) Conseguir más datos · b) Promediar varios modelos · c) Subir `max_depth` del árbol · d) Subir `k` en KNN

**3.** Si el ruido de los datos se duplica (de 0.3 a 0.6), el piso del error cuadrático…
- a) se duplica · b) se cuadruplica · c) no cambia · d) desaparece si usas un modelo más complejo

**4.** El tirador cuyos tiros caen todos juntos pero lejos del centro representa un modelo con…
- a) Sesgo bajo y varianza baja · b) Sesgo alto y varianza baja · c) Sesgo bajo y varianza alta · d) Sesgo alto y varianza alta

**5.** El bagging (promediar muchos árboles) funciona sobre todo porque…
- a) baja el sesgo · b) baja la varianza · c) elimina el ruido · d) hace al árbol más profundo

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Errores casi iguales (no hay brecha) y apenas mejores que una línea base: el modelo es demasiado rígido.
2. **c)** Subir `max_depth` hace al árbol **más** flexible: más varianza.
3. **b)** El piso es ruido² : 0.3² = 0.09 → 0.6² = 0.36, cuatro veces más.
4. **b)** Mira chueca, pulso firme: el tirador A.
5. **b)** Los temblores de cada árbol se cancelan al promediar; el sesgo casi no cambia (Ejercicio 7).
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un equipo tiene un modelo con varianza alta y propone dos soluciones: (1) pasar 3 meses recolectando el triple de datos, (2) usar un modelo más simple esta misma tarde. ¿Qué les dirías? ¿Hay algún caso en que la opción 1 sea claramente mejor?

**B. Caso de análisis 💼** Una cadena de cafeterías predice las ventas diarias de cada sucursal. Con una regresión lineal obtiene RMSE de 120 tazas en entrenamiento y 125 en validación. Un analista dice: *"El modelo generaliza perfecto, no hay brecha: no hay nada que mejorar."* ¿Estás de acuerdo? ¿Qué preguntarías o probarías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Primero la opción 2: es barata y rápida, y puede bastar. Pero un modelo más simple puede **subir el sesgo**; si el fenómeno es complejo, te quedarás corto. La opción 1 es mejor cuando el problema realmente necesita un modelo flexible (mucha interacción entre variables, patrones no lineales) y el sesgo del modelo simple es inaceptable: los datos extra bajan la varianza **sin** pagar en sesgo. Lo ideal es medirlo: entrena con el 25 %, 50 % y 100 % de los datos que ya tienes y mira si la validación sigue mejorando (curva de aprendizaje, Lección 4.4). Si sigue bajando, más datos ayudarán.

**B.** No necesariamente. **Sin brecha** solo descarta varianza alta; no dice nada del **sesgo**. Si una línea base (por ejemplo, "las ventas del mismo día de la semana pasada", Lección 4.7) logra RMSE de 110, el modelo es **peor** que no pensar: tiene sesgo alto. Preguntaría: ¿cuál es el RMSE de una línea base? ¿Cuánto vende una sucursal típica (120 tazas de error es mucho si vende 300, poco si vende 5,000)? Y probaría un modelo más flexible o mejores características (día de la semana, clima, feriados).
</details>

### 9.3 🧩 Práctica integradora: "¿Cuánta luz gastará el edificio?"

Una empresa de energía quiere predecir el **consumo eléctrico diario** de un edificio de oficinas según la **temperatura exterior**. Cuando hace frío se prende la calefacción y cuando hace calor el aire acondicionado, así que el consumo tiene forma de **U**. Como inventamos el fenómeno, podemos medir sesgo y varianza de verdad.

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira el fenómeno y una muestra de 40 días**

In [ ]:
def consumo_real(t):
    return 300 + 0.3 * (t - 20) ** 2          # kWh: mínimo a 20 °C

RUIDO_KWH = 50

def muestra_edificio(rng, n=40):
    t = rng.uniform(0, 40, n)
    return t, consumo_real(t) + rng.normal(0, RUIDO_KWH, n)

t_malla = np.linspace(2, 38, 50)

def descomponer_edificio(crear_modelo, n=40, repeticiones=200, semilla=0):
    rng = np.random.default_rng(semilla)
    P = np.array([crear_modelo().fit(t.reshape(-1, 1), y).predict(t_malla.reshape(-1, 1))
                  for t, y in (muestra_edificio(rng, n) for _ in range(repeticiones))])
    sesgo2 = np.mean((P.mean(axis=0) - consumo_real(t_malla)) ** 2)
    varianza = np.mean(P.var(axis=0))
    return sesgo2, varianza, sesgo2 + varianza + RUIDO_KWH**2

t, y = muestra_edificio(np.random.default_rng(1))
plt.scatter(t, y, c="k", s=20); plt.plot(t_malla, consumo_real(t_malla), "g--", label="verdad")
plt.xlabel("temperatura (°C)"); plt.ylabel("consumo (kWh)"); plt.legend(); plt.show()
print(f"Piso del ruido (ruido²): {RUIDO_KWH**2}")

**Paso 2 🔮 — Tres árboles.** Vamos a medir sesgo² y varianza de árboles de regresión con `max_depth` = **1, 3 y sin límite** (`None`). Antes de ejecutar, ordénalos de mayor a menor sesgo, y de mayor a menor varianza.

✍️ *Mi apuesta:*

In [ ]:
for d in [1, 3, None]:     # ✏️ prueba también 2, 4, 6
    s2, v, total = descomponer_edificio(lambda: DecisionTreeRegressor(max_depth=d, random_state=0))
    print(f"max_depth = {str(d):>4}  →  sesgo² = {s2:7.0f}   varianza = {v:6.0f}   error esperado = {total:6.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Sesgo: `max_depth=1` (una sola pregunta, "¿hace más de X grados?": dos escalones para dibujar una U) es el más alto con mucha diferencia (≈350). Los de profundidad 3 y sin límite tienen un sesgo² casi nulo (≈7 y ≈15; esa diferencia entre ellos es ruido de la simulación).
- Varianza: al revés. Sin límite es, de lejos, la más alta (≈2,500: cada hoja copia un solo día ruidoso), luego 3 (≈900) y la más baja es 1 (≈630).
- El mejor error esperado de los tres es `max_depth=3` (≈3,400); el árbol sin límite es el peor (≈5,000) aunque sea el que mejor "se ajusta" a cada muestra. Ninguno llega al piso de 2,500.
</details>

**Paso 3 ✏️ — Encuentra el fondo de la U.** Cambia la lista del paso anterior por `[1, 2, 3, 4, 5, 6, None]` y busca el `max_depth` con menor error esperado.

✍️ *Mi `max_depth` elegido:*

<details><summary><b>👀 Ver respuesta</b></summary>

El mínimo está en **`max_depth=2`** (≈3,290), con 3 muy cerca (≈3,410). Con 1 el sesgo todavía pesa (≈350 de sesgo²) y de 4 en adelante la varianza crece mucho más rápido de lo que baja el sesgo (≈3,830 con 4, ≈4,520 con 6). Es el mismo dibujo de la Sección 4, con otra perilla. Si dos valores casi empatan, quédate con el más simple.
</details>

**Paso 4 🔮 — ¿Y si tuviéramos un año de datos?** El gerente pregunta si vale la pena guardar **365 días** en vez de 40. Predice qué pasará con el sesgo² y la varianza del árbol **sin límite** y del de `max_depth=1`.

In [ ]:
for d in [1, None]:
    for n in [40, 365]:
        s2, v, total = descomponer_edificio(lambda: DecisionTreeRegressor(max_depth=d, random_state=0), n=n)
        print(f"max_depth = {str(d):>4}, n = {n:>3}  →  sesgo² = {s2:6.0f}   varianza = {v:6.0f}   error = {total:6.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Árbol de **una pregunta**: con 365 días su varianza baja a menos de la mitad (≈630 → ≈290), pero su sesgo² **no baja** (≈350 → ≈410; sigue siendo dos escalones para una U). **Más datos no curan el sesgo.**
- Árbol **sin límite**: ¡su varianza **no cambia** (≈2,490 → ≈2,500)! Un árbol sin límite **siempre** termina con hojas de un solo día, así que cada predicción copia el ruido de un día, tenga 40 o 365. Más datos solo ayudan a la varianza si el modelo los aprovecha para **promediar**.

Para el gerente: más datos ayudan, pero no son mágicos. Hace falta **también** controlar la complejidad.
</details>

**Paso 5 ✋ — Promediar.** La celda compara dos `BaggingRegressor` de 50 árboles cada uno, con los 40 días originales: uno promedia árboles **sin límite** y otro árboles de **`max_depth=3`**. ¿Alguno le gana al mejor árbol solo del paso 3 (≈3,290)?

In [ ]:
for d in [None, 3]:     # ✏️
    bolsa = lambda: BaggingRegressor(DecisionTreeRegressor(max_depth=d), n_estimators=50, random_state=0)
    s2, v, total = descomponer_edificio(bolsa, repeticiones=100)   # 100 repeticiones para ir más rápido
    print(f"50 árboles con max_depth = {str(d):>4}  →  sesgo² = {s2:4.0f}   varianza = {v:5.0f}   error esperado = {total:5.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Promediar árboles **sin límite** baja su varianza a menos de la mitad (≈2,490 → ≈1,190; error ≈5,000 → ≈3,700), pero **no** alcanza al mejor árbol solo: partía de demasiado lejos.
- Promediar árboles de **`max_depth=3`** baja su varianza de ≈900 a ≈520 y logra el **mejor error de toda la práctica (≈3,030)**, por debajo del mejor árbol solo.

Lección: promediar y controlar la complejidad **se combinan**. Ajustar a la vez la profundidad y el número de árboles es justo lo que harás con Random Forest en el Módulo 6.
</details>

**Paso 6 ✋ — Explícaselo al gerente.** En 3 frases y sin palabras técnicas, explica por qué "el modelo que mejor se ajusta a los datos históricos" (el árbol sin límite) **no** es el que recomiendas.

✍️ *Mi explicación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"El modelo más detallado aprende de memoria cada día del pasado, incluidas las rarezas que no se repetirán (un día con una fiesta, un sensor que falló). Por eso, si lo hubiéramos entrenado con otros 40 días, nos habría dado pronósticos muy distintos: no es confiable. Recomendamos uno un poco más sencillo, que capta la idea general —más consumo cuanto más se aleja la temperatura de 20 °C— y da pronósticos estables."*
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "Machine Learning Fundamentals: Bias and Variance" (inglés, con subtítulos).
- 📘 **Lectura:** Scott Fortmann-Roe, *Understanding the Bias-Variance Tradeoff* (ensayo clásico con la analogía de las dianas).
- 📘 **Libro:** *An Introduction to Statistical Learning* (James, Witten, Hastie, Tibshirani), sección 2.2.2, *The Bias-Variance Trade-Off*. Gratis en línea.
- 🔭 **Curiosidad:** en redes neuronales gigantes aparece el *double descent* (doble descenso): pasado cierto punto, añadir aún más complejidad vuelve a bajar el error. Es un tema de investigación activa; si te pica la curiosidad, busca *"Deep Double Descent"* (Nakkiran et al., 2019).

---
⬅️ **Anterior:** [4.4 Sobreajuste vs. subajuste: memorizar no es aprender](4.4-sobreajuste-vs-subajuste.ipynb) · ➡️ **Siguiente:** [4.6 Validación cruzada: k-fold, estratificada y temporal](4.6-validacion-cruzada.ipynb)